# 03 · Offline demo — the analysis, on synthetic data

**What this teaches:** what the CRAF'd Forecast API *computes* — posterior subsetting, HDI/MAP
summaries, and geographic aggregation — exercised **in-process** on a small synthetic
dataset, with **no network and no credentials**. It mirrors `01_quickstart`'s analytics but
runs anywhere.

**Audience:** anyone who wants to understand or reproduce the service's analytics without an
API key — reviewers, contributors, or a reader exploring the repo before requesting access.

> **How to run this notebook.** It uses only the public library (`views_crafdapi`) on
> **synthetic data with fictional geography** (`notebooks/_synthetic.py`, fixed seed) — no
> live API, no `.env`, no real CRAF'd data. A full *Run All* finishes in a few seconds.
>
> The live service serves a **global** forecast (every land cell worldwide) for historical
> **and** forecast — see `01`/`02` for that real coverage; this demo uses a toy lattice.

In [ ]:
import sys
from pathlib import Path

# make `_synthetic` importable whether the cwd is the repo root or notebooks/
for _p in (Path.cwd(), Path.cwd() / "notebooks"):
    if (_p / "_synthetic.py").exists():
        sys.path.insert(0, str(_p))
        break

import _synthetic as syn
from views_crafdapi.data.handlers import ForecastDataset

## 1 · A synthetic forecast dataset

`make_demo_dataframe()` returns a CRAF'd-shaped frame on an 8×8 **toy lattice** with fictional
countries (`WST`/`EST`) nesting cleanly into GAUL admin levels, and `n_samples` posterior
draws per cell for two targets. We wrap it in a `ForecastDataset` — the same class the
service builds from a real artifact.
> **Naming:** these synthetic targets use the real forecast *data-endpoint* names (`pred_lr_ged_sb`, `pred_lr_ged_ns`). The live **analysis** endpoint (`hdi_map`) further renames the collapsed columns to `sb_map`, `sb_hdi90_lower`, … — see the [data dictionary](../docs/api/data_dictionary.md). All values are raw fatality counts.


In [ ]:
df = syn.make_demo_dataframe(n_x=8, n_y=8, n_months=3, n_samples=256, seed=0)
ds = ForecastDataset(df)

print(f"cells x months : {df.shape[0]} rows")
print(f"sample size    : {ds.sample_size} draws per cell")
print(f"targets        : {ds.targets}")
print(f"geo levels     : {list(ds.levels)}")
df.head(3)

## 2 · Subsetting

`get_subset_dataframe` slices by time, entity, sample index, and geographic level — the
in-process equivalent of the API's `/subset` endpoints. Here: one country, one month,
aggregated.

In [ ]:
ds.get_subset_dataframe(entity_ids=["WST"], level="country", aggregate=True, time_ids=600)

## 3 · Posterior summaries — HDI and MAP

A forecast for a cell is not a number — it is a **posterior of draws**. `calculate_hdi_map`
collapses each posterior to a **MAP** point estimate and a **Highest-Density Interval**
(default 90%), plus the sample min/max. These are the numbers the API serves from
`/analysis/.../hdi-map`.

In [ ]:
hdi = ds.calculate_hdi_map(alpha=0.9, with_metadata=True)
cols = ["pred_lr_ged_sb_hdi90_lower", "pred_lr_ged_sb_map", "pred_lr_ged_sb_hdi90_upper"]
hdi[cols].head()

## 4 · Aggregation across geographic levels

Aggregating distributions is **not** the same as aggregating point estimates: the service
sums the *aligned posterior draws* across constituent cells *before* collapsing, so the
aggregate's uncertainty is honest (HDI of the sum ≠ sum of the HDIs). Note how the unit
count shrinks as we climb the hierarchy.

In [ ]:
for level in ["country", "gaul0", "gaul1", "gaul2"]:
    agg = ds.calculate_hdi_map(aggregate=True, level=level)
    print(f"{level:8s}: {agg.shape[0]:3d} (unit x month) rows")

# the country-level aggregate, with MAP and 90% HDI per country-month
ds.calculate_hdi_map(aggregate=True, level="country")[
    ["pred_lr_ged_sb_hdi90_lower", "pred_lr_ged_sb_map", "pred_lr_ged_sb_hdi90_upper"]
]

## 5 · A map view (toy lattice)

The cell-level MAP estimates carry the lattice coordinates, so `plot_pixel_grid` renders
them directly. Borders are off — these are **synthetic coordinates**, not real geography.

In [ ]:
import matplotlib.pyplot as plt
from views_crafdapi.plotting import plot_pixel_grid

month0 = ds._time_values.tolist()[0]
cell_map = ds.calculate_hdi_map(with_metadata=True).xs(month0, level="month_id")

fig, ax, _ = plot_pixel_grid(
    cell_map, value_col="pred_lr_ged_sb_map",
    transform="none", cmap="YlOrRd", borders=None, figsize=(5, 5),
)
ax.set_title("Synthetic MAP intensity — toy lattice (illustrative only)", fontsize=10)
plt.show()

## Next steps

- **`01_quickstart.ipynb`** — the same analytics over the **live HTTP API** (needs an API key).
- **`02_visualization.ipynb`** — richer maps and multi-panel comparisons via `CrafdApiClient`.
- API reference: [`docs/api/README.md`](../docs/api/README.md); authentication: ADR-027.